# Running & Inspecting Containers

## What's covered

- **The full anatomy of `docker run`** — every flag worth knowing, grouped by purpose
- **Foreground, detached, interactive** — what `-d`, `-i`, `-t`, and `-it` actually do
- **Naming, hostname, `--rm`** — controlling identity and cleanup
- **The container lifecycle in depth** — `create`, `start`, `stop`, `kill`, `restart`, `pause`/`unpause`, `rm`, and the states they transition between
- **Restart policies** — `no`, `on-failure`, `always`, `unless-stopped`, and how the daemon recovers containers after a host reboot
- **Resource limits** — `--memory`, `--memory-swap`, `--cpus`, `--cpu-shares`, `--pids-limit`, and the cgroup primitives underneath
- **Environment variables** — `-e`, `--env-file`, precedence
- **Logs** — `docker logs`, `-f`, `--since`, `--tail`, the logging drivers preview
- **Looking inside a running container** — `exec`, `attach`, `inspect`, `top`, `stats`, `diff`
- **Moving files in and out** — `docker cp`
- **Snapshotting** — `docker commit`, when (rarely) to reach for it
- **Live-updating limits** — `docker update`

By the end you should be able to launch any container with the right flags, debug a stuck one, watch what it's doing in real time, and explain every column in `docker ps`.

## The full anatomy of `docker run`

`docker run` is the busiest command in Docker. Its full shape:

```
docker run [OPTIONS] IMAGE[:TAG|@DIGEST] [COMMAND] [ARG...]
```

The options divide into a few mental buckets. We'll meet each bucket in turn.

| Bucket | Flags |
|---|---|
| **Lifecycle / attachment** | `-d`, `-i`, `-t`, `--rm`, `--name`, `--hostname` |
| **Networking** *(notebook 05)* | `-p`, `-P`, `--network`, `--dns`, `--add-host` |
| **Storage** *(notebook 04)* | `-v`, `--mount`, `--tmpfs`, `--read-only` |
| **Environment** | `-e`, `--env-file`, `-w` (workdir), `-u` (user) |
| **Resource limits** | `--memory`, `--memory-swap`, `--cpus`, `--cpu-shares`, `--pids-limit`, `--ulimit` |
| **Security** *(notebook 08)* | `--cap-add`, `--cap-drop`, `--security-opt`, `--user`, `--read-only` |
| **Restart & init** | `--restart`, `--init` |
| **Labels / metadata** | `--label` |

We covered networking and storage flags in passing in notebook 01; we'll spend whole notebooks on them later. This notebook is the deep dive on the other buckets.

## Foreground, detached, interactive — what `-d`, `-i`, `-t` actually do

These three flags get combined a lot, and people memorise the combinations without knowing what each one means. Let's separate them.

### `-d` — detached

Without `-d`, `docker run` attaches your terminal to the container's stdout and stderr. The container's output streams into your terminal, and `docker run` doesn't return until the container exits. Hitting Ctrl-C sends `SIGINT` to the container.

With `-d`, the container is detached — it runs in the background and `docker run` immediately returns the container's ID. The container's stdout/stderr is still captured (you can read it later with `docker logs`), just not streamed to your terminal.

- **Use `-d` for long-lived services** (web servers, databases) — you want them running in the background.
- **Skip `-d` for one-shot commands** (build steps, scripts) — you want to see the output and have `run` block until done.

### `-i` — interactive (keep stdin open)

Without `-i`, the container's stdin is closed immediately. Any program that tries to read from stdin will see EOF and exit. That's fine for `nginx`, broken for `bash`.

With `-i`, stdin stays open. Bytes you type in your terminal are piped to the container's stdin.

### `-t` — allocate a TTY

Without `-t`, the container has no terminal. It's like running a command via SSH with `-T`. Interactive prompts may look weird, line editing doesn't work, `clear` doesn't, colours might not appear.

With `-t`, Docker allocates a pseudo-terminal for the container. Now it looks and feels like a real terminal — prompts, colours, line editing all work.

### Why people write `-it` together

`-it` (combine `-i` and `-t`) is the *interactive shell* combo. Whenever you want to drop into a container as if you were SSH'd in, you use `-it`:

```bash
docker run -it --rm ubuntu:24.04 bash
docker run -it --rm python:3.12 python
docker exec -it my-running-container sh
```

Without `-i`, your typed input goes nowhere. Without `-t`, the shell prompt looks broken.

### Why `-d -it` is weird (but sometimes useful)

You usually don't combine `-d` and `-it`. But sometimes you want to start a long-lived container in the background that you'll `exec` into later, and you want it to *not* exit because its main process closed stdin. The idiom:

```bash
docker run -d -it --name sandbox ubuntu:24.04 bash
docker exec -it sandbox bash   # later, attach a shell
```

The `-d -it` keeps `bash` alive in the background waiting for stdin that's not coming. Cute, but `docker run -d ubuntu sleep infinity` is clearer.

In [ ]:
# Foreground, one-shot: blocks until done
!docker run --rm alpine:3.20 sh -c 'echo "hello $(uname -r)"; sleep 1; echo done'
!echo '---'
# Detached, long-lived: returns immediately
!docker run -d --rm --name detached-demo alpine:3.20 sh -c 'while true; do date; sleep 5; done'
!docker ps --filter name=detached-demo
!echo '---'
# Look at the logs
!sleep 6
!docker logs detached-demo
!docker stop detached-demo

## Naming, hostname, and `--rm`

Two flags control how the container is identified, and one controls whether it sticks around.

**`--name NAME`** — give the container a stable name you'll use to refer to it. If you don't, Docker generates one like `quirky_borg` or `vibrant_einstein` (the two-word "Docker name" generator). Auto-generated names are fine for throwaways; pick `--name` for anything you'll talk to more than once.

```bash
docker run -d --name web nginx:alpine
docker stop web   # much nicer than `docker stop 4f8a9c...`
```

Names are unique on a host. You can't have two containers named `web` even if one is stopped — you have to `docker rm web` first or pick a different name.

**`--hostname NAME`** — set the hostname *inside* the container (what `hostname` returns from within). By default this is the container's short ID. Useful for services that include their hostname in logs or registration.

**`--rm`** — automatically remove the container when it exits. Without it, every `docker run` leaves an `Exited` container around forever (until you `docker rm` it or `docker system prune`). With it, the container is garbage-collected on exit.

- **Use `--rm` for one-shot commands** (`docker run --rm -it ubuntu bash`) — you don't want graveyard containers cluttering `docker ps -a`.
- **Skip `--rm` for production services** — you might want the exit code / final logs of a crashed service for debugging.

**Trade-off:** `--rm` is incompatible with `--restart` (other than `--restart=no`). A container can't be both auto-removed *and* auto-restarted by Docker.

## The lifecycle in depth

Notebook 01 sketched the lifecycle. Here's the full state diagram and the commands that move a container between states.

```
                   docker create
   image  ------------------------>  created
                                        |
                                        | docker start
                                        v
                                     running <-----+
                                      |  |  |      |
                              docker  |  |  |      | docker start
                              pause   |  |  |      |
                                      v  |  |      |
                                   paused |  |   stopped
                                      |   |  |      ^
                              docker  |   |  +------|---- docker stop (SIGTERM,
                              unpause |   |         |                  wait, SIGKILL)
                                      +---+         |---- docker kill (SIGKILL)
                                                    |---- container exits on its own
                                        |           |
                                        | docker rm |
                                        v           |
                                      gone <--------+ (docker rm on stopped)
```

The commands:

- **`docker create IMAGE`** — set up a container but don't start it. Allocates the writable layer, assigns an ID, fills in `--name`, etc. You get a container ID back. Rarely used directly; `docker run` is `create` + `start` combined.
- **`docker start CONTAINER`** — start a created or stopped container. Keeps the existing writable layer, so any files written previously are still there.
- **`docker stop CONTAINER`** — graceful stop. Sends `SIGTERM`; if the process hasn't exited after 10s (configurable with `--time`), sends `SIGKILL`. This is what production deploys use.
- **`docker kill CONTAINER`** — `SIGKILL` immediately, no grace period. Use when `stop` won't work or you don't care about clean shutdown.
- **`docker restart CONTAINER`** — `stop` then `start`, in one call.
- **`docker pause CONTAINER`** / **`docker unpause CONTAINER`** — freezes all processes in the container's cgroup using the kernel's freezer. The processes are still in memory, just not scheduled. Useful for taking consistent snapshots or for debugging.
- **`docker rm CONTAINER`** — delete a stopped container, releasing its writable layer. Add `-f` to force-remove a running one (does `kill` + `rm`). Add `-v` to also remove any anonymous volumes the container created.

The corresponding inspection commands:

- **`docker ps`** — running. `-a` includes stopped. `-q` prints just IDs.
- **`docker ps --filter status=exited`** — only stopped containers.
- **`docker inspect CONTAINER`** — full JSON state.

In [ ]:
# Walk through the lifecycle explicitly
!docker create --name life-demo alpine:3.20 sh -c 'echo started; sleep 30; echo done'
!docker ps -a --filter name=life-demo --format 'table {{.Names}}\t{{.Status}}'
!echo '--- start ---'
!docker start life-demo
!sleep 1
!docker ps --filter name=life-demo --format 'table {{.Names}}\t{{.Status}}'
!echo '--- pause / unpause ---'
!docker pause life-demo
!docker ps --filter name=life-demo --format 'table {{.Names}}\t{{.Status}}'
!docker unpause life-demo
!echo '--- stop ---'
!docker stop life-demo
!docker ps -a --filter name=life-demo --format 'table {{.Names}}\t{{.Status}}'
!echo '--- logs ---'
!docker logs life-demo
!echo '--- remove ---'
!docker rm life-demo
!docker ps -a --filter name=life-demo --format 'table {{.Names}}\t{{.Status}}' || echo 'gone'

## Restart policies

By default a container that exits stays exited. Production services need to recover from crashes (and from host reboots). `--restart` controls that.

| Policy | Behaviour |
|---|---|
| **`no`** *(default)* | Never restart. |
| **`on-failure[:max-retries]`** | Restart if the container exits non-zero. Optional max retry count. |
| **`always`** | Restart whenever the container exits, *and* whenever the daemon starts. So after `systemctl restart docker` or a host reboot, the container comes back. Even if you `docker stop` it manually, the daemon will start it again on next boot. |
| **`unless-stopped`** | Like `always`, *except* if you explicitly `docker stop` it, it stays stopped across daemon restarts. The usual production choice. |

```bash
docker run -d --restart=unless-stopped --name api myorg/api:1.0
```

A few details that bite:

- **Restart policy applies after the first successful `start`.** A container that fails to start at all (bad image, syntax error) does not get retried.
- **Restart backoff is exponential**, capped at about a minute, to avoid hammering the host when a container is in a crash loop.
- **Restart counter resets** when the container has been running successfully for 10 seconds.
- **Restart and `--rm` are mutually exclusive.** Pick one.
- **Inspect a container** to see its policy and how many times it's restarted:

```bash
docker inspect --format '{{.HostConfig.RestartPolicy}} restarts={{.RestartCount}}' CONTAINER
```

In [ ]:
# Demonstrate restart policy: crash loop with on-failure:3
!docker run -d --name flaky --restart=on-failure:3 alpine:3.20 sh -c 'echo "starting"; exit 1'
!sleep 4
!docker inspect --format '{{.HostConfig.RestartPolicy.Name}}:{{.HostConfig.RestartPolicy.MaximumRetryCount}}  restarts={{.RestartCount}}  state={{.State.Status}}' flaky
!docker rm -f flaky

## Resource limits

By default, a container can consume **all** the host's memory and CPU. That's almost never what you want — one runaway container can take down the rest. Docker exposes the kernel's cgroup limits via a few `run` flags.

### Memory

- **`--memory=512m`** (or `1g`, `2g`) — hard cap on RAM. If the container exceeds it, the kernel's OOM killer kills the process. Container exits with code 137.
- **`--memory-swap=1g`** — total of memory + swap. Set equal to `--memory` to disable swap entirely (recommended for predictable behaviour). Set to `-1` to allow unlimited swap.
- **`--memory-reservation=256m`** — *soft* limit. Under memory pressure, the kernel tries to pull this container back to 256m, but doesn't kill it.
- **`--oom-kill-disable`** — disable the OOM killer for this container. **Don't.** It just shifts the OOM problem elsewhere on the host.

### CPU

- **`--cpus=1.5`** — equivalent to "1.5 CPUs worth of time." Implemented under the hood with `--cpu-period` and `--cpu-quota` (CFS bandwidth control). This is the easy, modern flag.
- **`--cpu-shares=512`** — relative weight (default 1024). Only matters when CPU is contended — a container with shares 512 gets half the CPU time of one with shares 1024 when they're both runnable.
- **`--cpuset-cpus="0,1"`** — pin to specific CPU cores. Useful for low-latency workloads that don't want to migrate between cores.

### Other limits

- **`--pids-limit=200`** — max number of PIDs in the container. Stops fork bombs.
- **`--ulimit nofile=65535:65535`** — set a ulimit (open file descriptors here). Same flag, several limits.

### Reading limits back

```bash
docker inspect --format 'memory={{.HostConfig.Memory}}  cpus={{.HostConfig.NanoCpus}}' CONTAINER
```

Memory is in bytes; `NanoCpus` is "billionths of a CPU," so `1500000000` means 1.5 CPUs.

In [ ]:
# Run a container with caps, see the limits land
!docker run -d --name limits-demo --memory=256m --cpus=0.5 --pids-limit=100 alpine:3.20 sh -c 'sleep 60'
!docker inspect --format 'Memory: {{.HostConfig.Memory}} bytes
NanoCpus: {{.HostConfig.NanoCpus}} (1.5 CPUs would be 1500000000)
PidsLimit: {{.HostConfig.PidsLimit}}' limits-demo
!docker rm -f limits-demo

## Environment variables — `-e` and `--env-file`

Containerised apps read configuration from environment variables (the 12-factor pattern). Docker has two ways to set them.

**Inline with `-e`** — one variable per flag:

```bash
docker run -e DATABASE_URL=postgres://... -e LOG_LEVEL=info myorg/api
```

**Bulk with `--env-file FILE`** — a file of `KEY=value` lines, comments with `#`:

```
# api.env
DATABASE_URL=postgres://user:pw@db:5432/app
LOG_LEVEL=info
SENTRY_DSN=https://...
```

```bash
docker run --env-file=api.env myorg/api
```

Precedence (lowest to highest):

1. Variables baked into the image with `ENV` in the Dockerfile
2. `--env-file` files (later files win over earlier)
3. `-e` flags (rightmost wins on duplicates)

A neat shorthand: `-e VAR` (no value) **inherits** the value from your host shell. Handy for passing tokens without echoing them on the command line:

```bash
export GITHUB_TOKEN=ghp_...
docker run -e GITHUB_TOKEN myorg/ci-tool
```

For **secrets**, env vars are widely used but not ideal — they leak via `docker inspect`, `/proc`, and crash dumps. Notebook 08 covers proper secrets (Compose secrets, Swarm secrets, BuildKit secret mounts).

## Logs

`docker logs CONTAINER` reads what the container's main process wrote to stdout and stderr. Crucial commands and flags:

- **`docker logs CONTAINER`** — print everything since the container started.
- **`docker logs -f CONTAINER`** — follow new output as it arrives (`tail -f` style). Ctrl-C detaches.
- **`docker logs --tail=100 CONTAINER`** — last 100 lines.
- **`docker logs --since=10m CONTAINER`** — only entries from the last 10 minutes. Accepts `1h`, `2024-01-15T10:00:00`, etc.
- **`docker logs --until=1h CONTAINER`** — entries up to 1 hour ago (intersect with `--since` to window).
- **`docker logs -t CONTAINER`** — prefix each line with the daemon's timestamp.

### Where do logs actually live?

The daemon writes logs to a file managed by the active **logging driver**. The default driver is `json-file` and the file lives at `/var/lib/docker/containers/<container-id>/<container-id>-json.log`. `docker logs` reads from there.

Other drivers ship logs elsewhere — `journald`, `syslog`, `awslogs`, `gelf`, `fluentd`, `splunk`. You set the driver globally in `/etc/docker/daemon.json` or per-container with `--log-driver`. Notebook 10 covers configuration; for now just know **`docker logs` only works for `json-file` and `journald`** — if you switch drivers, you read logs from the destination instead.

### Two gotchas

1. **Buffered output.** Many runtimes (Python especially) buffer stdout when it's not a terminal — your prints can take ages to appear in `docker logs`. Set `PYTHONUNBUFFERED=1` or use `python -u`.
2. **Disk fill.** `json-file` driver grows unbounded unless you configure rotation:

```json
// /etc/docker/daemon.json (or per-container with --log-opt)
{
  "log-driver": "json-file",
  "log-opts": { "max-size": "10m", "max-file": "3" }
}
```

Without this, a chatty container can fill your disk in hours.

## Looking inside a running container — `exec`, `attach`, `inspect`, `top`, `stats`, `diff`

The container's running and you want to see what's going on. Six tools.

### `docker exec -it CONTAINER CMD` — run a command inside

The single most-used debugging command. Drops you (or a script) into a running container without affecting its main process.

```bash
docker exec -it web sh         # interactive shell
docker exec web ls /app        # one-shot command
docker exec -u root web bash   # as a specific user
docker exec -e DEBUG=1 web ./tool   # with extra env
```

The exec'd process is a sibling of the container's main process, not a child. When you exit, the container keeps running.

### `docker attach CONTAINER` — connect to the main process

Hooks your terminal up to the container's existing stdin/stdout/stderr. Unlike `exec`, you're not spawning a new process — you're talking to PID 1. Ctrl-C sends `SIGINT` to PID 1, which usually kills the container.

To detach without killing: `Ctrl-P Ctrl-Q` (the Docker detach sequence). Or run `docker attach --sig-proxy=false` so your Ctrl-C doesn't get forwarded.

Use `attach` to see *current* stdout in real time without `docker logs -f`'s buffering. Use `exec` for everything else.

### `docker inspect CONTAINER` — JSON state dump

The full state of the container: image, command, args, env, mounts, network config, restart policy, resource limits, current state, exit code, OOM-killed flag, started/finished timestamps, IP address.

```bash
docker inspect web
docker inspect --format '{{.State.Status}}' web
docker inspect --format '{{.NetworkSettings.IPAddress}}' web
docker inspect --format '{{json .Config.Env}}' web | jq
```

`--format` uses Go template syntax. Once you know it, it's faster than piping to `jq`.

### `docker top CONTAINER` — what processes are running

Like `ps` but scoped to the container.

```bash
docker top web
docker top web aux   # any ps flags work
```

### `docker stats` — live resource usage

```bash
docker stats           # all running containers, live updating
docker stats web db    # specific containers
docker stats --no-stream   # one snapshot, no live update
```

Shows CPU %, memory usage and limit, network I/O, block I/O, PIDs.

### `docker diff CONTAINER` — what files changed in the writable layer

Shows what the container has added (`A`), changed (`C`), or deleted (`D`) relative to its image.

```bash
docker diff web
```

Useful for spotting "the app wrote to its own filesystem" — usually a sign it should be writing to a volume instead.

In [ ]:
# Spin up nginx, look inside it
!docker run -d --name introspect-demo -p 8090:80 nginx:alpine
!sleep 1
!echo '--- exec a shell command ---'
!docker exec introspect-demo nginx -v
!echo '--- inspect, formatted ---'
!docker inspect --format 'image={{.Config.Image}}  status={{.State.Status}}  pid={{.State.Pid}}  ip={{.NetworkSettings.IPAddress}}' introspect-demo
!echo '--- top ---'
!docker top introspect-demo
!echo '--- one-shot stats ---'
!docker stats --no-stream introspect-demo
!echo '--- diff (write a file inside, then check) ---'
!docker exec introspect-demo sh -c 'echo hello > /tmp/marker'
!docker diff introspect-demo
!docker rm -f introspect-demo

## Moving files in and out — `docker cp`

`docker cp` copies files or directories between the container and the host. Works on running *and* stopped containers.

```bash
# Host -> container
docker cp ./config.yaml web:/app/config.yaml

# Container -> host
docker cp web:/var/log/nginx/access.log ./access.log

# Whole directory (use a trailing dot like cp -r semantics)
docker cp web:/etc/nginx/. ./nginx-conf/
```

A few details:

- The container path is **absolute**. Host path can be relative.
- If the destination doesn't exist, it's created. If it exists as a directory, the source is placed inside it.
- `docker cp` doesn't follow symlinks by default — use `-L`.
- Permissions/ownership are preserved on the source.
- This is **not** how you do persistent storage. For data that should survive container removal, use volumes (notebook 04). `docker cp` is for ad-hoc operations: pulling out a log, dropping in a config to test.

### A different model: `docker exec -i` with `tar`

For piped transfers (often used in CI), you can stream tar through stdin/stdout:

```bash
tar -cf - ./config | docker exec -i web tar -xf - -C /app
docker exec web tar -cf - /var/log | tar -xf - -C ./host-logs
```

This pattern is what `docker cp` does internally.

## Snapshotting — `docker commit`

`docker commit CONTAINER NEW_IMAGE_NAME` takes a snapshot of the container's current filesystem and creates a new image from it.

```bash
docker run -it --name lab ubuntu:24.04 bash
# inside: apt update && apt install -y python3 vim
exit
docker commit lab my-ubuntu-with-python
docker run -it --rm my-ubuntu-with-python bash   # has python and vim
```

This works. **But you should almost never use it.** The result is opaque — there's no record of *how* the image was built, no Dockerfile to review, no diff against the base. It's the "save game" of Docker images: convenient, but you've lost the recipe.

**When commit is OK:**

- **Exploration** — you're poking at an image, want to save state for later before you destroy the container.
- **Capturing a forensic snapshot** of a crashed container for debugging.

**When commit is not OK:**

- Anything you'll need to rebuild later.
- Anything you'll share with a teammate.
- Anything that goes to a registry for production.

For those, write a Dockerfile. Future-you and future-teammates will thank present-you.

## Live-updating limits — `docker update`

For an already-running container, you can change *some* runtime settings without restarting it. `docker update` is the command.

```bash
docker update --memory=1g --cpus=2 web
docker update --restart=unless-stopped web
```

What you can change at runtime:

- Resource limits: `--memory`, `--memory-swap`, `--cpus`, `--cpu-shares`, `--cpuset-cpus`, `--cpu-period`, `--cpu-quota`, `--pids-limit`
- Restart policy: `--restart`
- Block I/O weight: `--blkio-weight`

What you **cannot** change at runtime:

- Networking (port mappings, network mode, hostname)
- Volumes / mounts
- Environment variables
- Image
- Entrypoint, command, user

For anything in the second list, you have to stop, remove, and re-create the container.

The use case for `docker update`: "this container is using too much memory and getting OOM-killed; let me raise the limit without losing its state." Or: "I want to flip restart policy to `unless-stopped` so it survives a daemon restart.